# Generation of LCA indicators and associated .mod and .dat files

In [1]:
# %pip install brightway2
# %pip install mescal
# %pip install energyscope

In [2]:
%load_ext autoreload
%autoreload 2

In [3]:
import sys
from pathlib import Path

In [4]:
NOTEBOOK_DIR = Path.cwd()
LCA_PROJECTS_ROOT = NOTEBOOK_DIR.parent.parent
sys.path.insert(1, str(LCA_PROJECTS_ROOT / '02_Regionalization' / '01_Notebooks'))

In [5]:
import os
import pandas as pd
import bw2data as bd
from mescal import *
from energyscope.models import Model
from energyscope.energyscope import Energyscope
from energyscope.result import postprocessing
from utils import (
    DATA_DIR,
    add_biogenic_climate_change_to_impact_scores_df,
    add_rhhd_and_reqd_to_impact_scores_df,
)
from shared.utils import run_model, load_snapshot

In [6]:
ei_version = '3.12'
year = 2050
ssp_rcp = 'SSP2-RCP26'
iam = 'tiam-ucl'

The assessment type `base_wo_iam` and the column 'IAM assumptions' only applies when `year` is set to 2050.

In [7]:
LCA_RESULTS_DIR = f'../03_Results/LCA/{year}/{iam}/{ssp_rcp}/'
AMPL_FILES_DATA_DIR = f'../02_AMPL_files/data/{year}/{iam}/{ssp_rcp}/'
AMPL_FILES_MODEL_DIR = f'../02_AMPL_files/model/'

## Initialize the EnergyScope model

In [8]:
# AMPL licence 
path_to_ampl_licence = r'C:\Users\matth\ampl' # Path to the AMPL license file
os.environ['PATH'] = path_to_ampl_licence+':'+os.environ['PATH']

In [9]:
# Initialize the 2023 QC model with .mod and .dat files
model = load_snapshot(year)

In [10]:
# Solve the model and get results
results = run_model(model)

Gurobi 12.0.0: 

## Load data and initialize the ESM class

In [11]:
# Load the data
mapping = pd.read_csv('./Data/mapping.csv')
unit_conversion = pd.read_excel('./Data/unit_conversion.xlsx')  # open file and press enter in one computation cell to avoid misreading
techno_compositions = pd.read_csv(DATA_DIR / 'technology_compositions.csv')
tech_specifics = pd.read_csv('./Data/technology_specifics.csv')
efficiency = pd.read_csv(DATA_DIR / 'efficiency.csv')
lifetime = pd.read_csv(DATA_DIR / 'lifetime.csv')
mapping_es_flows_to_cpc = pd.read_csv(DATA_DIR / 'mapping_esm_flows_to_CPC.csv')
impact_abbrev = pd.read_csv(DATA_DIR / 'impact_abbrev.csv')
mapping_new_product_to_cpc = pd.read_csv(DATA_DIR / 'mapping_new_products_to_CPC.csv')

In [12]:
# Load the model from energyscope model
model = results.parameters['layers_in_out'].reset_index().rename(columns={'index0': 'Name', 'index1': 'Flow', 'layers_in_out': 'Amount'}).drop(columns=['Run'])
model = model[model['Amount'] != 0]
# model.to_csv(DATA_DIR / f'model_{year}.csv', index=False)

In [13]:
# Set up your Brightway project
bd.projects.set_current(f'ecoinvent{ei_version}')

In [14]:
# Database names
name_main_database = f'ecoinvent_cutoff_{ei_version}_{iam}_{ssp_rcp}_{year}'
name_biosphere_db = 'biosphere3'
name_spatialized_biosphere_db = 'biosphere3_spatialized_flows'
name_es_database = f'EnergyScope_CA-QC_{iam}_{ssp_rcp}_{year}'

In [15]:
regionalize_foregrounds = ['Operation', 'Resource']

In [16]:
main_db = Database(db_names=name_main_database, create_pickle=True)
# spatialized_biosphere_db = Database(db_names=name_spatialized_biosphere_db)

Getting activity data


100%|██████████| 44097/44097 [00:00<00:00, 89159.94it/s] 


Adding exchange data to activities


100%|██████████| 1437481/1437481 [01:21<00:00, 17568.86it/s]


Filling out exchange data


100%|██████████| 44097/44097 [00:07<00:00, 6144.71it/s] 
2026-09-04 16:46:18,520 - Database - INFO - Loaded ecoinvent_cutoff_3.12_tiam-ucl_SSP2-RCP26_2050 from brightway!


ecoinvent_cutoff_3.12_tiam-ucl_SSP2-RCP26_2050.pickle created!


In [17]:
ranking_best_ecoinvent_locations_for_QC = [
    'CA-QC', # Quebec
    'CAN', # Canada in IMAGE and TIAM-UCL
    'CA', # Canada
    'NAM', # North America in MESSAGE
    'CAZ', # Canada - Australia - New Zealand in REMIND
    'RNA', # North America
    'US', # United States
    'USA', # United States in REMIND and IMAGE
    'CA-AB', # Alberta
    'GLO', # Global average 
    'RoW', # Rest of the world
]

In [18]:
# Add CPC categories to the main database
main_db.add_CPC_categories(mapping_new_products_to_CPC=mapping_new_product_to_cpc, overwrite_existing_CPC=True)

In [19]:
# Change the main database name if needed
if mapping['Database'].iloc[0] != name_main_database:
    mapping['Database'] = name_main_database

In [20]:
esm = ESM(
    # Mandatory inputs
    mapping=mapping,
    unit_conversion=unit_conversion,
    model=model,
    mapping_esm_flows_to_CPC_cat=mapping_es_flows_to_cpc,
    main_database=main_db,
    esm_db_name=name_es_database,
    
    # Optional inputs
    technology_compositions=techno_compositions,
    tech_specifics=tech_specifics,
    lifetime=lifetime,
    efficiency=efficiency,
    regionalize_foregrounds=regionalize_foregrounds,
    accepted_locations=['CA-QC'],
    locations_ranking=ranking_best_ecoinvent_locations_for_QC,
    esm_location='CA-QC',
    results_path_file=LCA_RESULTS_DIR,
    biosphere_db_name=name_biosphere_db,
    
    # If we want regionalized results 
    # spatialized_biosphere_db=spatialized_biosphere_db,
)

In [21]:
esm.clean_inputs()

In [22]:
# Adapt mapping file to ESM location
esm.change_location_mapping_file()
esm.mapping.to_csv('./Data/mapping_with_loc.csv', index=False)

In [23]:
missing_flows = main_db.test_mapping_file(esm.mapping)

2026-09-04 16:46:45,909 - Database - INFO - Mapping successfully linked to the database


In [24]:
missing_flows

[]

In [25]:
esm.check_inputs()

2026-09-04 16:46:46,686 - Mescal - WARNING - List of technologies or resources that are in the model file but not in the mapping file. Their impact scores will be set to the default value: ['CARBON_MINERALIZATION', 'CO2_E', 'DEC_RENOVATION', 'DHN_RENOVATION', 'DIESEL_S', 'ELEC_EXPORT_EHV', 'ELEC_EXPORT_HV', 'ELEC_EXPORT_LV', 'ELEC_EXPORT_MV', 'ELEC_S', 'ELEC_STO', 'GASOLINE_S', 'HT_LT', 'LT_DEC_WH', 'LT_DHN_WH', 'NG_S', 'RES_GEO', 'RES_HYDRO', 'RES_SOLAR', 'RES_TIDAL', 'RES_WIND_OFFSHORE', 'RES_WIND_ONSHORE', 'SNG_S', 'STO_CO2', 'STO_DIE', 'STO_ELEC', 'STO_GASO', 'STO_H2', 'STO_NG', 'STO_SNG']
2026-09-04 16:46:46,894 - Mescal - WARNING - List of technologies or resources that are in the mapping file but not in the model file (this will not be a problem in the workflow): ['BATTERY', 'DEC_TH_STORAGE', 'DHN_TH_STORAGE', 'EHP_H2_GRID', 'EHP_NG_GRID', 'EHV_GRID', 'ELEC_EXPORT', 'HP_H2_GRID', 'HP_NG_GRID', 'HV_GRID', 'LP_H2_GRID', 'LP_NG_GRID', 'LV_GRID', 'MP_H2_GRID', 'MP_NG_GRID', 'MV_GRID

In [26]:
main_db = {}  # Free memory

### Generate ESM database

In [ ]:
# Foreground regionalization, double-counting removal, and efficiency harmonization
esm.create_esm_database()

## Generate LCA metrics

In [ ]:
methods = [
    'IMPACT World+ Midpoint 2.2.1_regionalized for ecoinvent v3.12',  # also works with non-spatialized datasets
    'IMPACT World+ Damage 2.2.1_regionalized for ecoinvent v3.12',  # also works with non-spatialized datasets
    'IMPACT World+ Damage 2.2.1 for ecoinvent v3.12 (incl. CO2 uptake)',
    'IMPACT World+ Midpoint 2.2.1 for ecoinvent v3.12 (incl. CO2 uptake)',
]

### Life-cycle emissions

In [ ]:
contrib_analysis = None  # 'emissions', 'processes', 'both' or None

In [ ]:
# LCIA, Lifetime harmonization
if contrib_analysis is not None:
    R_long, contrib_analysis_res, _ = esm.compute_impact_scores(
        methods=methods,
        impact_abbrev=impact_abbrev,
        contribution_analysis=contrib_analysis,
        contribution_analysis_limit_type='number',
        contribution_analysis_limit=30,
    )
    contrib_analysis_emissions = contrib_analysis_res[contrib_analysis_res.database.str.contains('biosphere')]
    contrib_analysis_processes = contrib_analysis_res[~contrib_analysis_res.database.str.contains('biosphere')]
    contrib_analysis_emissions.to_csv(f'{LCA_RESULTS_DIR}contribution_analysis_emissions.csv', index=False)
    contrib_analysis_processes.to_csv(f'{LCA_RESULTS_DIR}contribution_analysis_processes.csv', index=False)
else:
    R_long, contrib_analysis_res, _ = esm.compute_impact_scores(
        methods=methods,
        impact_abbrev=impact_abbrev,
    )

In [ ]:
# Set the impact of transformers to zero for operation
R_long['Value'] = R_long.apply(lambda row: 0 if row['Name'].startswith('TRAFO_') and row['Type'] == 'Operation' else row['Value'], axis=1)

In [ ]:
R_long.to_csv(f'{LCA_RESULTS_DIR}impact_scores.csv', index=False) # [impact / kW(h) or pkm(/h) or tkm(/h)]

### Territorial carbon emissions

In [ ]:
if esm.esm_db is None:
    esm.esm_db = Database(esm.esm_db_name)

In [ ]:
_, contrib_analysis_all_processes, _ = esm.compute_impact_scores(
    methods=methods,
    specific_lcia_abbrev=['m_CCS_all'],
    impact_abbrev=impact_abbrev,
    contribution_analysis='processes',
    contribution_analysis_limit_type='number',
    contribution_analysis_limit=2000,
)

In [ ]:
contrib_analysis_all_processes.to_csv(f'{LCA_RESULTS_DIR}contribution_analysis_all_processes_ccst.csv', index=False)

### Add a remaining AoP category (total AoP - climate change) and biogenic CC

In [ ]:
# To skip the impact assessment step
R_long = pd.read_csv(f'{LCA_RESULTS_DIR}impact_scores.csv')
R_long_direct_emissions = pd.read_csv(f'{LCA_RESULTS_DIR}impact_scores_direct_emissions.csv')
contrib_analysis_all_processes = pd.read_csv(f'{LCA_RESULTS_DIR}contribution_analysis_all_processes_ccst.csv')
impact_abbrev = pd.read_csv(DATA_DIR / 'impact_abbrev.csv')

In [ ]:
R_long, _ = add_biogenic_climate_change_to_impact_scores_df(R_long, impact_abbrev)

In [ ]:
R_long_direct_emissions, _ = add_biogenic_climate_change_to_impact_scores_df(R_long_direct_emissions, impact_abbrev)

In [ ]:
R_long, impact_abbrev = add_rhhd_and_reqd_to_impact_scores_df(R_long, impact_abbrev)

In [ ]:
R_long_direct_emissions, _ = add_rhhd_and_reqd_to_impact_scores_df(R_long_direct_emissions, impact_abbrev)

## Create the .mod and .dat files

In [ ]:
metadata = {
    'ecoinvent_version': ei_version,
    'year': year,
    'iam': iam,
    'ssp_rcp': ssp_rcp,
}

In [ ]:
specific_lcia_abbrev = ['RHHD', 'REQD', 'm_CCS_all']

In [ ]:
# Snapshot model taken as a 1-step transition model
to_remove = ['TRAIN_FREIGHT_H2_HYBRID_ELD', 'TRAIN_FREIGHT_H2_HYBRID_LD', 'ELEC_EXPORT']
esm.pathway = True
R_long['Year'] = 2025 if year==2023 else 2020
R_long_direct_emissions['Year'] = 2025 if year==2023 else 2020
contrib_analysis_all_processes['Year'] = 2025 if year==2023 else 2020
R_long = R_long[~R_long['Name'].isin(to_remove)]
R_long_direct_emissions = R_long_direct_emissions[~R_long_direct_emissions['Name'].isin(to_remove)]
contrib_analysis_all_processes = contrib_analysis_all_processes[~contrib_analysis_all_processes['act_name'].isin(to_remove)]

In [ ]:
# Create .dat file
esm.normalize_lca_metrics(
    R=R_long,
    mip_gap=1e-6,
    lcia_methods=methods,
    specific_lcia_abbrev=specific_lcia_abbrev,
    impact_abbrev=impact_abbrev,
    path=AMPL_FILES_DATA_DIR,
    metadata=metadata,
    file_name='QC_techs_lca',
)

In [ ]:
# Create .dat file for territorial emissions
esm.normalize_lca_metrics(
    assessment_type='territorial emissions',
    R=R_long,
    contrib_processes=contrib_analysis_all_processes,
    mip_gap=1e-6,
    lcia_methods=methods,
    specific_lcia_abbrev=['m_CCS_all'],
    impact_abbrev=impact_abbrev,
    path=AMPL_FILES_DATA_DIR,
    metadata=metadata,
    file_name='QC_techs_lca_territorial',
)

In [ ]:
# Create the .mod file
esm.generate_mod_file_ampl(
    lcia_methods=methods,
    impact_abbrev=impact_abbrev,
    specific_lcia_abbrev=specific_lcia_abbrev,
    path=AMPL_FILES_MODEL_DIR,
    metadata=metadata,
    file_name='QC_objectives_lca',
)

In [ ]:
# Create the .mod file for territorial and abroad emissions
esm.generate_mod_file_ampl(
    assessment_type='territorial emissions',
    lcia_methods=methods,
    impact_abbrev=impact_abbrev,
    specific_lcia_abbrev=['m_CCS_all'],
    path=AMPL_FILES_MODEL_DIR,
    metadata=metadata,
    file_name='QC_objectives_lca_territorial',
)